# 20장. 오분류 분석

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 18장 (계수 해석), 19장 |
| 이 장의 결과물 | 오분류 목록(`misclassified.txt`), 영어·한국어 각 5건 이상의 원인 분석표, 길이별·주석자 일치별 정확도 |
| 과제 연결 | "오분류된 문서 5개 이상의 원인을 분석한다 (BoW 한계와 연결하여 서술)" |
| 관련 학습 목표 | ④ BoW가 문맥과 어순을 반영하지 못하는 한계를 구체적 사례로 설명 |

**학습 목표**
- 틀린 문서를 모으고, 확신도가 높은 오답부터 살펴볼 수 있다.
- 단어별 기여도(계수 × TF-IDF)로 "어떤 단어 때문에 이렇게 예측했는지" 보여 줄 수 있다.
- 오분류 원인을 유형으로 나누고, 각 사례를 BoW의 한계와 연결해 서술할 수 있다.
- 문서 길이와 레이블 모호성처럼 모델 밖의 원인을 숫자로 확인할 수 있다.

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

# ── 8~10장에서 만든 전처리 파이프라인 (src/preprocess.py로 옮길 내용) ──
from konlpy.tag import Okt
from nltk.corpus import stopwords

PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")
HANJA_ONLY = re.compile(r"^[一-鿿]+$")
KO_KEEP_POS = {"Noun", "Verb", "Adjective", "Alpha"}
EN_STOP = frozenset(stopwords.words("english"))
DOMAIN_STOPWORDS_EN = frozenset({
    "would", "could", "might", "may", "also", "like", "get", "got", "go", "going", "know", "think",
    "see", "say", "make", "want", "look", "one", "well", "much", "many", "even", "way", "something",
    "anyone", "since", "still", "really", "right", "another", "cant",
})
STOPWORDS_KO = frozenset({
    "하다", "되다", "있다", "없다", "이다", "않다",
    "등", "것", "수", "중", "및", "때", "후", "내", "듯", "제", "개", "의", "로", "종합",
})
_okt = Okt()


def clean_en(text: str) -> str:
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


def clean_ko(text: str) -> str:
    return " ".join(KO_CLEAN.sub(" ", PLACEHOLDER.sub(" ", text)).split())


class EnglishPreprocessor:
    def __init__(self, use_domain_stopwords: bool = True, min_len: int = 2, max_len: int = 20):
        self.stopwords = EN_STOP | (DOMAIN_STOPWORDS_EN if use_domain_stopwords else frozenset())
        self.min_len, self.max_len = min_len, max_len

    def __call__(self, text: str) -> list[str]:
        tokens = []
        for tok in clean_en(text).split():
            if tok in self.stopwords:
                continue
            tok = tok.removesuffix("'s").replace("'", "")
            if self.min_len <= len(tok) <= self.max_len and tok not in self.stopwords:
                tokens.append(tok)
        return tokens


class KoreanPreprocessor:
    def __init__(self, tokenizer: str = "okt", use_stopwords: bool = True):
        if tokenizer not in ("okt", "whitespace"):
            raise ValueError(f"tokenizer는 'okt' 또는 'whitespace': {tokenizer}")
        self.tokenizer = tokenizer
        self.stopwords = STOPWORDS_KO if use_stopwords else frozenset()

    def tokenize(self, text: str) -> list[str]:
        if self.tokenizer == "whitespace":
            return clean_ko(text).lower().split()
        return [w.lower() for w, p in _okt.pos(clean_ko(text), norm=True, stem=True)
                if p in KO_KEEP_POS or (p == "Foreign" and HANJA_ONLY.match(w))]

    def __call__(self, text: str) -> list[str]:
        return [t for t in self.tokenize(text) if t not in self.stopwords and not t.isdigit()]

# ── 로드 + 전처리를 한 번에 (src/corpus.py로 옮길 내용) ──
MIN_TOKENS = {"en": 5, "ko": 2}   # 전처리 후 토큰이 이보다 적은 문서는 뺀다 (짧은 답글, 주제어 없는 제목)
MIN_DF = {"en": 3, "ko": 2}       # 어휘에 넣을 최소 문서 빈도 (11장)


def load_corpus(lang: str, tokenizer: str = "okt") -> pd.DataFrame:
    """열: text, label, tokens. lang은 "en" 또는 "ko"."""
    df = load_english() if lang == "en" else load_korean()
    prep = EnglishPreprocessor() if lang == "en" else KoreanPreprocessor(tokenizer)
    df["tokens"] = [prep(t) for t in df["text"]]
    df = df[df["tokens"].map(len) >= MIN_TOKENS[lang]].reset_index(drop=True)
    df.attrs["preprocessor"] = prep
    return df

# ── 13장에서 만든 NumPy TF-IDF (src/tfidf.py로 옮길 내용) ──
class NumpyTfidf:
    """TF(raw 또는 log) × smooth IDF, 문서마다 L2 정규화. sklearn TfidfVectorizer와 같은 정의."""

    def __init__(self, tf: str = "raw", smooth_idf: bool = True, norm: str | None = "l2",
                 min_df: int = 1, max_df: float = 1.0):
        if tf not in ("raw", "log"):
            raise ValueError(f"tf는 'raw' 또는 'log': {tf}")
        if norm not in ("l2", None):
            raise ValueError(f"norm은 'l2' 또는 None: {norm}")
        self.tf, self.smooth_idf, self.norm = tf, smooth_idf, norm
        self.min_df, self.max_df = min_df, max_df

    def fit(self, docs: list[list[str]]) -> "NumpyTfidf":
        n = len(docs)
        df: dict[str, int] = {}
        for doc in docs:
            for t in set(doc):
                df[t] = df.get(t, 0) + 1
        max_count = self.max_df * n if isinstance(self.max_df, float) else self.max_df
        terms = sorted(t for t, c in df.items() if self.min_df <= c <= max_count)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.n_docs_ = n
        self.df_ = np.array([df[t] for t in terms], dtype=np.float64)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n) / (1.0 + self.df_)) + 1.0
        else:
            self.idf_ = np.log(n / self.df_) + 1.0
        return self

    def count_matrix(self, docs: list[list[str]]) -> np.ndarray:
        rows, cols = [], []
        for i, doc in enumerate(docs):
            for t in doc:
                j = self.vocabulary_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        counts = np.zeros((len(docs), len(self.vocabulary_)), dtype=np.float64)
        np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
        return counts

    def transform(self, docs: list[list[str]]) -> np.ndarray:
        x = self.count_matrix(docs)
        if self.tf == "log":
            mask = x > 0
            x[mask] = 1.0 + np.log(x[mask])
        x *= self.idf_                                   # (n, V) * (V,) 브로드캐스팅
        if self.norm == "l2":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            x /= np.where(norms > 0, norms, 1.0)          # 빈 문서는 0 벡터로 둔다
        return x

    def fit_transform(self, docs: list[list[str]]) -> np.ndarray:
        return self.fit(docs).transform(docs)

# ── 17장에서 만든 분할·분류 코드 (src/classify.py로 옮길 내용) ──
from types import SimpleNamespace

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


def split(corpus: pd.DataFrame, seed: int = RANDOM_STATE) -> tuple[pd.DataFrame, pd.DataFrame]:
    train, test = train_test_split(corpus, test_size=0.2, stratify=corpus["label"], random_state=seed)
    return train.reset_index(drop=True), test.reset_index(drop=True)


def run_classification(corpus: pd.DataFrame, lang: str, seed: int = RANDOM_STATE, C: float = 1.0):
    """Train으로만 TF-IDF를 fit하고, 로지스틱 회귀를 학습해 Test로 평가한다."""
    train, test = split(corpus, seed)
    vec = NumpyTfidf(min_df=MIN_DF[lang])
    X_train = vec.fit_transform(train["tokens"].tolist())
    X_test = vec.transform(test["tokens"].tolist())
    model = LogisticRegression(C=C, max_iter=2000, random_state=RANDOM_STATE).fit(X_train, train["label"])
    y_pred = model.predict(X_test)
    return SimpleNamespace(
        train=train, test=test, vec=vec, model=model, X_train=X_train, X_test=X_test,
        y_pred=y_pred, labels=list(model.classes_), proba=model.predict_proba(X_test),
        accuracy=accuracy_score(test["label"], y_pred),
        f1_macro=f1_score(test["label"], y_pred, average="macro"),
    )

en = load_corpus("en")
ko = load_corpus("ko")
res_en = run_classification(en, "en")
res_ko = run_classification(ko, "ko")
for name, r in [("en", res_en), ("ko-okt", res_ko)]:
    n_wrong = int((r.y_pred != r.test["label"].to_numpy()).sum())
    print(f"{name:<7} 정확도 {r.accuracy:.3f}, Test {len(r.test):,}건 중 오분류 {n_wrong}건")

en      정확도 0.900, Test 930건 중 오분류 93건
ko-okt  정확도 0.770, Test 1,400건 중 오분류 322건


## 20.1 틀린 문서 모으기

틀린 문서마다 예측 카테고리의 확률(확신도)과 정답 카테고리의 확률을 붙여 표로 만듭니다. **확신도가 높은데 틀린 문서**부터 보는 것이 좋습니다. 모델이 "강한 단서"라고 믿은 것이 무엇인지가 가장 분명하게 드러나기 때문입니다.

In [2]:
def misclassified_table(r) -> pd.DataFrame:
    wrong = np.flatnonzero(r.y_pred != r.test["label"].to_numpy())
    rows = []
    for row in wrong:
        pred, true = r.y_pred[row], r.test.at[row, "label"]
        rows.append({"row": row, "true": true, "pred": pred,
                     "p_pred": r.proba[row, r.labels.index(pred)],
                     "p_true": r.proba[row, r.labels.index(true)],
                     "n_tokens": len(r.test.at[row, "tokens"]),
                     "text": " ".join(r.test.at[row, "text"].split())[:60]})
    return pd.DataFrame(rows).sort_values("p_pred", ascending=False).reset_index(drop=True)


wrong_en = misclassified_table(res_en)
wrong_ko = misclassified_table(res_ko)
print(f"영어 오분류 {len(wrong_en)}건 중 확신도 0.5 이상: {(wrong_en['p_pred'] >= 0.5).sum()}건")
print(f"한국어 오분류 {len(wrong_ko)}건 중 확신도 0.5 이상: {(wrong_ko['p_pred'] >= 0.5).sum()}건\n")
wrong_ko.head(10).round(2)

영어 오분류 93건 중 확신도 0.5 이상: 17건
한국어 오분류 322건 중 확신도 0.5 이상: 45건



,row,true,pred,p_pred,p_true,n_tokens,text
0,904,세계,정치,0.89,0.07,7,두테르테 필리핀 대통령 방한…내일 문대통령과 정상회담
1,382,경제,IT과학,0.83,0.05,8,LGU 마블 히어로 캐릭터 적용 AI스피커 출시
2,1345,경제,IT과학,0.82,0.06,9,고동진 사장 갤럭시S8 판매 갤럭시S7보다 15% 많아
3,1241,스포츠,정치,0.82,0.04,8,올림픽 문 대통령 뒤에 북한 옆에 미국은 빈자리
4,1192,정치,세계,0.82,0.05,7,문제는 백악관의 트럼프가 아니라 우리 내면의 트럼프
5,938,세계,정치,0.78,0.06,8,에티오피아 총리 北 무책임한 행동…우리는 한국편
6,193,경제,IT과학,0.75,0.04,7,SKT버라이즌 5G 공동연구 손 잡는다
7,432,경제,IT과학,0.74,0.12,10,LG전자 100kW급 태양광 발전용 올인원 ESS 출시
8,862,스포츠,세계,0.74,0.17,10,유럽축구 수장 트럼프 행정명령 美월드컵 개최에 도움안돼
9,1010,세계,스포츠,0.72,0.05,8,포틀랜드 워싱턴 꺾고 NBA 서부콘퍼런스 선두 도약


## 20.2 단어별 기여도 ⭐

18장에서 본 것처럼 로지스틱 회귀의 카테고리 $c$ 점수는

$$s_c = \sum_t w_{c,t}\, x_t + b_c$$

입니다. 그러니 $w_{c,t}\, x_t$가 바로 **단어 $t$가 카테고리 $c$ 점수에 기여한 양**입니다. 예측 카테고리 쪽 기여 단어와 정답 카테고리 쪽 기여 단어를 나란히 보면 "무엇이 판정을 뒤집었는지" 알 수 있습니다.

먼저 기여도를 모두 더하면 정말 점수가 되는지 확인합니다.

In [3]:
row = int(wrong_en.loc[3, "row"])
c = res_en.labels.index(res_en.y_pred[row])
contrib = res_en.model.coef_[c] * res_en.X_test[row]
print("기여도 합 + 절편 :", round(float(contrib.sum() + res_en.model.intercept_[c]), 6))
print("decision_function:", round(float(res_en.model.decision_function(res_en.X_test[row:row + 1])[0, c]), 6))

기여도 합 + 절편 : 1.91619
decision_function: 1.91619


In [4]:
def top_contributions(r, row: int, label: str, k: int = 5) -> list[tuple[str, float]]:
    c = r.labels.index(label)
    contrib = r.model.coef_[c] * r.X_test[row]
    idx = np.argsort(-contrib)[:k]
    return [(r.vec.terms_[j], round(float(contrib[j]), 2)) for j in idx if contrib[j] > 0]


def describe_case(r, row: int, text_len: int = 300, extra: str = "") -> str:
    pred, true = r.y_pred[row], r.test.at[row, "label"]
    text = " ".join(r.test.at[row, "text"].split())
    lines = [f"예측: {pred} ({r.proba[row, r.labels.index(pred)]:.2f}) / 실제: {true} "
             f"({r.proba[row, r.labels.index(true)]:.2f}), 토큰 {len(r.test.at[row, 'tokens'])}개{extra}",
             f"  문서: \"{text[:text_len]}{'…' if len(text) > text_len else ''}\"",
             f"  토큰: {r.test.at[row, 'tokens'][:20]}",
             f"  예측 쪽 기여: {top_contributions(r, row, pred)}",
             f"  정답 쪽 기여: {top_contributions(r, row, true) or '(없음)'}"]
    return "\n".join(lines)


def find_row(r, prefix: str) -> int:
    """문서 앞부분으로 Test 행 번호를 찾는다 (사례를 안정적으로 고정하려고)."""
    hits = [i for i, t in enumerate(r.test["text"]) if " ".join(t.split()).startswith(prefix)]
    assert len(hits) == 1, (prefix, hits)
    return hits[0]


print(describe_case(res_en, row))

예측: comp.graphics (0.69) / 실제: comp.windows.x (0.16), 토큰 35개
  문서: "Check into the vopl and vogle libraries. I beleive that I still have them available on ftp.c-mols.siu.edu but I'm not sure. I haven't seen any new announcements for the software but I'm sure it's still around. I have used it to do real-time data display and analysis as well as just for producing gra…"
  토큰: ['check', 'vopl', 'vogle', 'libraries', 'beleive', 'available', 'ftp', 'mols', 'siu', 'edu', 'sure', 'seen', 'new', 'announcements', 'software', 'sure', 'around', 'used', 'real', 'time']
  예측 쪽 기여: [('graphics', 0.6), ('formats', 0.19), ('software', 0.19), ('available', 0.18), ('ftp', 0.17)]
  정답 쪽 기여: [('display', 0.28), ('libraries', 0.12), ('tried', 0.1), ('software', 0.07), ('works', 0.07)]


## 20.3 원인 유형

사례를 볼 때 아래 유형 중 무엇에 해당하는지 붙여 두면 정리하기 쉽습니다. 한 사례에 여러 유형이 겹치기도 합니다.

| 유형 | 설명 | 연결되는 BoW 한계 |
|---|---|---|
| **주제 겹침** | 두 카테고리가 같은 어휘를 공유한다 | 단어 빈도만으로는 경계를 그을 수 없다 |
| **다의어·동음이의어** | 같은 글자가 다른 뜻 (`windows`, `문`) | 단어 하나 = 차원 하나, 뜻을 구분하지 못한다 |
| **문맥·의도 필요** | 단어는 A 주제인데 글의 목적은 B, 비유, 부정 | 어순·문맥을 버린다 |
| **레이블 모호** | 사람도 판단이 갈린다 | 모델 한계가 아니라 데이터 한계 |
| **전처리 부산물** | 정제·토큰화가 만든 조각 (`5G` → `g`) | 토큰 단위 설계의 한계 |
| **짧은 문서** | 단서 단어가 거의 없다 | 0이 아닌 차원이 적어 단어 하나에 좌우 |

## 20.4 사례 분석

### 영어

In [5]:
EN_CASES = [('E1', 'Look in the /pub/SPACE directory'), ('E2', "I'm looking for any leads to the source of a good Windows Meta File"), ('E3', 'Hello, I hope this is the right group to post this in.'), ('E4', 'How do I view .eps files on X?'), ('E5', 'What, did you leave the room each of the 100 or so times'), ('E6', 'And you know what? There is no such method inherent in real life either.'), ('E7', 'If anyone knows of such a site could they please send it to me also.')]

for name, prefix in EN_CASES:
    print(f"[{name}] " + describe_case(res_en, find_row(res_en, prefix)), "\n")

[E1] 예측: sci.space (0.85) / 실제: comp.graphics (0.10), 토큰 22개
  문서: "Look in the /pub/SPACE directory on ames.arc.nasa.gov - there are a number of earth images there. You may have to hunt around the subdirectories as things tend to be filed under the mission (ie, "APOLLO") rather than under the image subject."
  토큰: ['pub', 'space', 'directory', 'ames', 'arc', 'nasa', 'gov', 'number', 'earth', 'images', 'hunt', 'around', 'subdirectories', 'things', 'tend', 'filed', 'mission', 'ie', 'apollo', 'rather']
  예측 쪽 기여: [('space', 0.83), ('nasa', 0.48), ('earth', 0.45), ('mission', 0.31), ('things', 0.15)]
  정답 쪽 기여: [('image', 0.51), ('images', 0.35), ('number', 0.19), ('directory', 0.12), ('ie', 0.09)] 

[E2] 예측: comp.windows.x (0.55) / 실제: comp.graphics (0.36), 토큰 21개
  문서: "I'm looking for any leads to the source of a good Windows Meta File converter or interpreter. I need this for use outside the Windows environment. PD sources preferred, but not a requirement. Please reply to the address 

In [6]:
# E3, E4 확인용: 정답 단서가 될 만한 단어가 어휘에 있는지, 계수는 얼마인지
for word in ["startx", "x", "windows", "graphics"]:
    j = res_en.vec.vocabulary_.get(word)
    if j is None:
        print(f"{word!r}: 어휘에 없음")
    else:
        coefs = {(l.split(".", 1)[1] if l.startswith("comp.") else l.split(".")[-1]): round(float(w), 2) for l, w in zip(res_en.labels, res_en.model.coef_[:, j])}
        print(f"{word!r}: 문서 빈도 {int(res_en.vec.df_[j])}, 계수 {coefs}")

'startx': 문서 빈도 6, 계수 {'graphics': -0.06, 'windows.x': 0.14, 'baseball': -0.02, 'hockey': -0.02, 'space': -0.03}
'x': 어휘에 없음
'windows': 문서 빈도 198, 계수 {'graphics': 0.47, 'windows.x': 2.34, 'baseball': -0.95, 'hockey': -0.77, 'space': -1.08}
'graphics': 문서 빈도 252, 계수 {'graphics': 4.49, 'windows.x': -0.61, 'baseball': -1.16, 'hockey': -1.06, 'space': -1.66}


| # | 실제 → 예측 | 문서 (요약) | 원인 | BoW 한계 |
|---|---|---|---|---|
| E1 | graphics → **space** | NASA 서버의 `/pub/SPACE` 폴더에 지구 **이미지**가 많다는 안내 | **문맥·의도.** 글의 목적은 "이미지 파일 위치 안내"(그래픽)인데 단어 대부분이 `space`, `nasa`, `earth`, `mission` 같은 우주 어휘다. 정답 쪽 단서 `image`, `images`보다 우주 단어의 합이 컸다. | 단어가 "무엇에 대한 것"인지가 아니라 몇 번 나왔는지만 본다 |
| E2 | graphics → **windows.x** | Windows Meta File 변환기를 찾는다 | **다의어.** 여기서 `Windows`는 마이크로소프트 Windows인데, 모델에게 `windows`는 X Window 게시판의 강한 단서(계수가 크다)다. | 같은 글자는 같은 차원이라 뜻을 구분하지 못한다 |
| E3 | windows.x → **graphics** | Linux에서 `startx`를 하면 모니터가 **graphics mode**가 되고 흰 사각형만 나온다 | **다의어와 드문 단어.** `graphics`는 "그래픽 모드(화면 상태)"라는 뜻으로 쓰였다. 정답 단서 `startx`(X 실행 명령)는 문서 빈도가 아주 낮아 windows.x 계수가 작다(위 셀). | 드문 단어는 학습 근거가 적어 강한 신호가 되지 못한다 |
| E4 | windows.x → **graphics** | X에서 `.eps` 파일을 어떻게 보나 | **주제 겹침 + 전처리 부산물.** `image`, `postscript`, `files`는 그래픽 게시판 어휘다. 정답의 핵심인 "on **X**"의 `x`는 한 글자라 10장의 길이 필터로 지워져 어휘에 없다(위 셀). | 토큰 단위 설계가 핵심 단어를 지울 수 있다 |
| E5 | hockey → **baseball** | 야구 경기가 없다고 그렇게 말했는데 못 들었냐, 대신 하키를 보여 준다는데… | **언급과 주제의 구분 실패.** 하키 중계 이야기지만 "야구가 없어서"라는 맥락으로 `baseball`이 반복되어 `hockey`보다 큰 기여를 했다. | "무엇이 없다"와 "무엇에 대해 말한다"를 구분하지 못한다 (부정·문맥) |
| E6 | baseball → **space** | 테레사 수녀와 스탈린… "지구의 삶은 **팀 플레이**다" (스레드에서 벗어난 철학 논쟁) | **비유와 문맥 손실.** `earth`, `planet`, `life`, `world`가 우주 쪽으로 기여했지만 비유적 표현이다. 야구 스레드라는 맥락은 인용문을 지우면서 사라졌다(7장). | 어순·문맥을 버리고, 비유를 구분하지 못한다 |
| E7 | baseball → **graphics** | 그런 사이트를 아는 분은 저에게도 보내 주세요 | **짧은 문서.** 전처리 후 토큰이 5개뿐이고 주제어가 없다. "사이트를 알려 달라"는 표현이 FTP 사이트를 묻는 comp 게시판에 많아 graphics로 갔다. | 단서가 적으면 일반적인 표현 하나에 판정이 좌우된다 |

### 한국어
한국어는 KLUE 주석자 3명의 1순위 판단도 함께 봅니다(7장). 원본 파일에서 문서 ID(`guid`)로 찾습니다.

In [7]:
ynat_raw = pd.read_json(YNAT_PATH).set_index("guid")
first_scope = {g: ynat_raw.at[g, "annotations"]["annotations"]["first-scope"] for g in res_ko.test["guid"]}

KO_CASES = [('K1', '두테르테 필리핀 대통령 방한'), ('K2', '고동진 사장 갤럭시S8 판매'), ('K3', '올림픽 문 대통령 뒤에 북한 옆에'), ('K4', '문제는 백악관의 트럼프가 아니라'), ('K5', 'SKT버라이즌 5G 공동연구'), ('K6', 'KT AI사업 기자간담회'), ('K7', '양파 농장 가던 버스 논으로 추락')]
for name, prefix in KO_CASES:
    row = find_row(res_ko, prefix)
    annot = " · ".join(first_scope[res_ko.test.at[row, "guid"]])
    print(f"[{name}] " + describe_case(res_ko, row, extra=f", 주석자 1순위: {annot}"), "\n")

[K1] 예측: 정치 (0.89) / 실제: 세계 (0.07), 토큰 7개, 주석자 1순위: 세계 · 정치 · 세계
  문서: "두테르테 필리핀 대통령 방한…내일 문대통령과 정상회담"
  토큰: ['두테르테', '필리핀', '대통령', '방한', '내일', '대통령', '정상회담']
  예측 쪽 기여: [('대통령', 3.05), ('정상회담', 0.71), ('내일', 0.26)]
  정답 쪽 기여: [('대통령', 0.69), ('필리핀', 0.64), ('정상회담', 0.07)] 

[K2] 예측: IT과학 (0.82) / 실제: 경제 (0.06), 토큰 9개, 주석자 1순위: IT과학 · 경제 · 경제
  문서: "고동진 사장 갤럭시S8 판매 갤럭시S7보다 15% 많아"
  토큰: ['고동진', '사장', '갤럭시', 's', '판매', '갤럭시', 's', '보다', '많다']
  예측 쪽 기여: [('s', 1.39), ('갤럭시', 1.16), ('고동진', 0.25), ('보다', 0.2), ('많다', 0.12)]
  정답 쪽 기여: [('판매', 0.36), ('s', 0.13), ('사장', 0.12), ('갤럭시', 0.1)] 

[K3] 예측: 정치 (0.82) / 실제: 스포츠 (0.04), 토큰 8개, 주석자 1순위: 스포츠 · 스포츠 · 정치
  문서: "올림픽 문 대통령 뒤에 북한 옆에 미국은 빈자리"
  토큰: ['올림픽', '문', '대통령', '뒤', '북한', '옆', '미국', '빈자리']
  예측 쪽 기여: [('대통령', 1.73), ('북한', 1.02), ('문', 0.79), ('뒤', 0.17)]
  정답 쪽 기여: [('올림픽', 0.76), ('북한', 0.05), ('미국', 0.0)] 

[K4] 예측: 세계 (0.82) / 실제: 정치 (0.05), 토큰 7개, 주석자 1순위: 정치 · 정치 · 세계
  문서: "문제는 백악관의 트럼프가 아니라 우리 내면의 트럼프"
  토큰: ['문제', '백악관', 

| # | 실제 → 예측 | 제목 | 원인 | BoW 한계 |
|---|---|---|---|---|
| K1 | 세계 → **정치** | 두테르테 필리핀 대통령 방한…내일 문대통령과 정상회담 | **같은 단어, 다른 주어.** `대통령`이 두 번 나오고 정치 쪽 기여가 압도적이다. 한국 대통령이면 정치, 외국 대통령이면 세계인데 누구의 대통령인지 구분하지 못한다. 주석자 판단도 갈렸다. | 단어 간 관계(누가, 누구의)를 버린다 |
| K2 | 경제 → **IT과학** | 고동진 사장 갤럭시S8 판매 갤럭시S7보다 15% 많아 | **관점 + 전처리 부산물.** 판매량 기사(경제)지만 제품명 `갤럭시`와 `S8`에서 남은 `s`가 IT 쪽으로 강하게 기여했다. 핵심어 `판매`의 경제 쪽 기여는 작다. 주석자도 갈렸다. | 같은 대상을 어떤 관점으로 다루는지 반영하지 못한다 |
| K3 | 스포츠 → **정치** | 올림픽 문 대통령 뒤에 북한 옆에 미국은 빈자리 | **주제 혼합 + 다의어.** 올림픽 개막식 좌석 이야기지만 정치 단어(`대통령`, `북한`, `문`)가 여럿이고 스포츠 단어는 `올림픽` 하나다. `문`(대통령 성)은 "문(door)"과 같은 토큰이다. | 동음이의어를 구분하지 못한다 |
| K4 | 정치 → **세계** | 문제는 백악관의 트럼프가 아니라 우리 내면의 트럼프 | **비유와 부정.** 국내 정치를 비유로 비판하는 칼럼이다. "트럼프가 **아니라**"의 부정은 `아니다` 토큰 하나로 남아 오히려 정답 쪽에 약하게 기여했을 뿐, 의미는 반영되지 않는다. | 부정과 비유, 어순을 볼 수 없다 |
| K5 | 경제 → **IT과학** | SKT버라이즌 5G 공동연구 손 잡는다 | **관용 표현 + 전처리 부산물.** "손 잡는다"(제휴)가 경제 기사의 핵심인데 `손`, `잡다`로 쪼개져 의미를 잃었다. `5G`는 숫자가 떨어져 `g`만 남아 IT 쪽 단서가 됐다. | 여러 단어가 모여 만드는 뜻(관용구)을 잃는다 |
| K6 | 사회 → **IT과학** | KT AI사업 기자간담회 | **짧은 문서 + 레이블 모호.** 토큰이 4개뿐이고 `kt`, `ai`가 IT 쪽으로 강하다. 기자간담회라는 형식이 사회로 분류된 이유는 제목만으로는 알기 어렵고, 주석자 의견도 갈렸다. | 단서가 적으면 강한 단어 하나에 좌우된다 |
| K7 | 사회 → **세계** | 양파 농장 가던 버스 논으로 추락…1명 사망·11명 부상종합2보 | **문맥 + 전처리 부산물.** 사고 기사는 국내(사회)와 해외(세계) 모두에 있어서 `사망`, `추락` 같은 단어로는 장소를 알 수 없다. `종합2보`에서 남은 `보`와 `명`도 기여했다. 주석자 3명은 모두 사회로 일치했으니 사람에게는 쉬운 문제다. | 장소·주체 같은 문맥을 보지 못한다 |

> 사례마다 위 셀 출력(기여 단어와 숫자, 주석자 판단)을 근거로 원인을 썼습니다. 직접 분석할 때도 **추측이 아니라 기여도 출력**을 근거로 쓰는 것이 중요합니다.

## 20.5 모델 밖의 원인 확인

### 문서 길이별 정확도

In [8]:
def accuracy_by_length(r, bins: list[int]) -> pd.DataFrame:
    n_tok = r.test["tokens"].map(len).to_numpy()
    correct = r.y_pred == r.test["label"].to_numpy()
    groups = pd.cut(n_tok, bins=bins, right=False)
    g = pd.DataFrame({"토큰 수": groups, "correct": correct}).groupby("토큰 수", observed=True)["correct"]
    return pd.DataFrame({"Test 문서": g.size(), "정확도": g.mean().round(3)})


print("영어")
print(accuracy_by_length(res_en, [5, 10, 20, 50, 100, 10**6]).to_string())
print("\n한국어 (Okt)")
print(accuracy_by_length(res_ko, [2, 5, 7, 9, 11, 100]).to_string())

영어
                Test 문서    정확도
토큰 수                          
[5, 10)              65  0.754
[10, 20)            164  0.799
[20, 50)            334  0.898
[50, 100)           202  0.970
[100, 1000000)      165  0.976

한국어 (Okt)
           Test 문서    정확도
토큰 수                     
[2, 5)          43  0.767
[5, 7)         226  0.774
[7, 9)         532  0.716
[9, 11)        498  0.811
[11, 100)      101  0.842


영어는 토큰이 적을수록 정확도가 뚜렷하게 떨어집니다. 짧은 글은 벡터에서 0이 아닌 차원이 몇 개 없어 단어 하나에 판정이 좌우되기 때문입니다(E7). 한국어는 길이 구간에 따라 정확도가 오르내릴 뿐 영어처럼 한 방향의 뚜렷한 경향은 없습니다. 제목이 모두 짧아서 길이 차이 자체가 작기 때문입니다. 한국어에서는 길이보다 다음의 레이블 모호성이 더 큰 원인입니다.

### 한국어: 주석자 의견 일치 여부별 정확도

In [9]:
scopes = [first_scope[g] for g in res_ko.test["guid"]]
unanimous = np.array([len(set(s)) == 1 for s in scopes])
correct = res_ko.y_pred == res_ko.test["label"].to_numpy()
wrong_rows = np.flatnonzero(~correct)
chosen = np.mean([res_ko.y_pred[i] in scopes[i] for i in wrong_rows])

print(f"Test 제목 {len(correct):,}건 중 주석자 3명의 1순위가 일치한 제목: {unanimous.mean():.1%}")
print(f"정확도 - 의견 일치 제목: {correct[unanimous].mean():.3f} / 의견이 갈린 제목: {correct[~unanimous].mean():.3f}")
print(f"오분류 {len(wrong_rows)}건 중 모델의 예측을 주석자 중 한 명 이상이 골랐던 비율: {chosen:.1%}")

Test 제목 1,400건 중 주석자 3명의 1순위가 일치한 제목: 61.9%
정확도 - 의견 일치 제목: 0.873 / 의견이 갈린 제목: 0.603
오분류 322건 중 모델의 예측을 주석자 중 한 명 이상이 골랐던 비율: 42.2%


- 주석자 의견이 갈린 제목에서 정확도가 크게 떨어집니다. **사람도 하나로 정하기 어려운 문서**에서 모델도 틀립니다.
- 오분류의 상당 부분은 모델의 예측을 주석자 중 누군가는 골랐던 것입니다. 이런 경우는 "모델이 틀렸다"기보다 "정답이 하나가 아니다"에 가깝습니다.
- 그래서 한국어 정확도를 영어와 단순 비교하면 안 됩니다. 한국어는 제목이 짧고(평균 8개 토큰) 레이블 자체가 모호한 문서가 많습니다.

## 📝 리포트용 정리: 오분류 원인과 BoW 한계

| BoW 한계 | 영어 사례 | 한국어 사례 |
|---|---|---|
| 문맥·의도·비유를 보지 못함 | E1, E5, E6 | K4, K7 |
| 다의어·동음이의어 | E2, E3 | K3 (`문`) |
| 단어 사이의 관계(주어·관점)를 버림 | | K1, K2 |
| 부정을 반영하지 못함 | E5 | K4 |
| 여러 단어가 만드는 뜻(관용구)을 잃음 | | K5 |
| 토큰 단위 설계의 부산물 | E4 (`x` 삭제) | K2 (`s`), K5 (`g`), K7 (`보`) |
| 짧은 문서 | E7 (길이별 정확도) | K6 |
| 데이터 한계 (레이블 모호) | | K1, K2, K6 (주석자 불일치) |

리포트에는 사례 표(20.4)와 이 요약표, 길이별·주석자 일치별 정확도를 함께 넣습니다. 21장에서 이 한계들을 일부러 만든 예문으로 다시 보여 줍니다.

## ✍️ 실습: 직접 사례 분석하기

한국어 오분류 중 **실제 `정치` → 예측 `세계`**인 문서를 확신도 높은 순으로 3건 골라 `describe_case`로 출력하세요(주석자 판단도 함께). 그리고 각 사례의 원인 유형(20.3)과 BoW 한계를 한 줄씩 적어 보세요.

In [10]:
# 여기에 작성하세요

**정답 (코드)**

In [11]:
cases = wrong_ko[(wrong_ko["true"] == "정치") & (wrong_ko["pred"] == "세계")].head(3)
for row in cases["row"]:
    annot = " · ".join(first_scope[res_ko.test.at[row, "guid"]])
    print(describe_case(res_ko, int(row), extra=f", 주석자 1순위: {annot}"), "\n")

예측: 세계 (0.82) / 실제: 정치 (0.05), 토큰 7개, 주석자 1순위: 정치 · 정치 · 세계
  문서: "문제는 백악관의 트럼프가 아니라 우리 내면의 트럼프"
  토큰: ['문제', '백악관', '트럼프', '아니다', '우리', '내면', '트럼프']
  예측 쪽 기여: [('트럼프', 2.66), ('백악관', 0.34), ('우리', 0.08)]
  정답 쪽 기여: [('문제', 0.24), ('아니다', 0.16)] 

예측: 세계 (0.40) / 실제: 정치 (0.36), 토큰 8개, 주석자 1순위: 정치 · 정치 · 정치
  문서: "대북 소식통 中북한식당 나머지 종업원 중 귀순대기자 없는듯"
  토큰: ['대북', '소식통', '中', '북한', '식당', '나머지', '종업원', '기자']
  예측 쪽 기여: [('中', 1.37), ('식당', 0.21), ('종업원', 0.08), ('대북', 0.03)]
  정답 쪽 기여: [('북한', 0.87), ('대북', 0.63), ('종업원', 0.13)] 

예측: 세계 (0.36) / 실제: 정치 (0.24), 토큰 11개, 주석자 1순위: 세계 · 정치 · 정치
  문서: "대북제재 속 평양 상품전에 中업체 100여개 참가 성황"
  토큰: ['대북', '제재', '속', '평양', '상품', '전', '中', '업체', '여개', '참가', '성황']
  예측 쪽 기여: [('中', 1.11), ('속', 0.27), ('제재', 0.24), ('전', 0.07), ('대북', 0.02)]
  정답 쪽 기여: [('평양', 0.54), ('대북', 0.51), ('제재', 0.27)] 



**분석 예시:** 정치 → 세계 오분류는 대부분 `트럼프`, `美`, `中`, 외국 지명처럼 **세계 쪽 계수가 큰 외국 관련 단어**가 국내 정치 기사에 등장해서 생깁니다. 원인 유형은 "주제 겹침"과 "단어 사이의 관계를 버림"(한국 정부가 외국에 대해 말하는가, 외국이 한국에 대해 말하는가)입니다. 출력의 기여 단어와 주석자 판단을 근거로, 사례마다 이렇게 한 줄씩 정리하면 됩니다.

## 🧐 퀴즈

**Q1.** 오분류 분석을 확신도가 높은 오답부터 보는 이유는?
<details><summary>▶ 정답 보기</summary>

모델이 강하게 믿은 단서가 무엇인지 가장 분명하게 드러나기 때문입니다. 확신도가 낮은 오답은 두 카테고리 사이에서 애매했던 경우가 많아 원인이 덜 뚜렷합니다.
</details>

**Q2.** 단어별 기여도 $w_{c,t} x_t$를 모두 더하면 무엇이 되나요? 이것이 분석에 왜 유용한가요?
<details><summary>▶ 정답 보기</summary>

절편 $b_c$를 빼면 카테고리 $c$의 점수 $s_c$가 됩니다. 점수를 단어별로 정확히 나눠 볼 수 있으므로 "어떤 단어가 얼마나 판정을 이끌었는지"를 추측이 아니라 숫자로 설명할 수 있습니다.
</details>

**Q3.** K3 사례에서 `문`(대통령 성)이 정치 쪽으로 기여한 것은 BoW의 어떤 한계인가요?
<details><summary>▶ 정답 보기</summary>

동음이의어 문제입니다. "문 대통령"의 `문`과 "문을 열다"의 `문`이 같은 토큰(같은 차원)이라 뜻을 구분하지 못합니다.
</details>

**Q4.** 한국어 오분류 중 모델의 예측을 주석자 중 누군가가 골랐던 사례가 많다는 것은 무엇을 뜻하나요?
<details><summary>▶ 정답 보기</summary>

그 오분류의 상당수는 레이블이 하나로 정해지기 어려운 문서라는 뜻입니다. 모델 개선만으로 없앨 수 없는 데이터 자체의 한계이고, 정확도의 현실적인 상한을 낮춥니다.
</details>

## ✅ 체크포인트
- [ ] 오분류 목록을 만들고 확신도 순으로 정렬할 수 있다.
- [ ] 단어별 기여도로 예측의 근거를 보여 줄 수 있다.
- [ ] 영어와 한국어 각각 5건 이상을 과제 형식(예측/실제/문서/원인/한계)으로 분석할 수 있다.
- [ ] 각 원인을 BoW의 구체적인 한계와 연결할 수 있다.
- [ ] 문서 길이와 레이블 모호성의 영향을 숫자로 보여 줄 수 있다.

## 🔗 과제 연결과 다음 장
- 사례 표와 요약표는 README "한계와 개선 방향"의 오분류 분석에 씁니다(학습 목표 ④). 전체 오분류 목록은 프로젝트에서 `outputs/<데이터>/misclassified.txt`로 저장합니다.
- Part 6(분류 실험)이 끝났습니다. **21장. BoW의 한계**에서 이 장의 사례들을 일부러 만든 예문으로 재현하고, 개선 방향을 정리합니다.